# 2. Multi-agent / Sub-agent (Advanced) — 제품 탑재형 AI: Fleet Telemetry 분석 시나리오
**Main Agent(감독) ↔ Data Analyst(Python) ↔ Writer ↔ QA Reviewer**

---

## 🧩 시나리오(가상)
스마트 생활가전/로보틱스 제품이 대량으로 배포되었을때, 아래 같은 분석을 수행합니다.

- 특정 모델에서 **에러코드/고장률**이 급증하는지
- 지역/펌웨어 버전별로 **이상 패턴**이 있는지
- 고객지원/현장 기사 대응을 위해 **근거 기반 리포트**를 빠르게 작성해야 하는지

이 노트북은 더미 텔레메트리 데이터를 만들고,
**LLM 멀티에이전트로 (1)분석 → (2)리포트 작성 → (3)QA 검수** 흐름을 실습합니다.


In [1]:
# 맨 처음 한 번만 실행하세요. (패키지 없으면 create_agent 등 import 실패)
!pip -q install -U langchain-openai langchain-core langchain langgraph pandas numpy langchain-experimental pydantic

ERROR: Could not install packages due to an OSError: [Errno 13] Permission denied: '/opt/conda/lib/python3.10/site-packages/typing_extensions-4.15.0.dist-info/licenses/LICENSE'
Consider using the `--user` option or check the permissions.



In [2]:
import os, getpass, json
import pandas as pd
import numpy as np

from langchain.chat_models import init_chat_model
from langchain_openai import ChatOpenAI
from langchain_experimental.tools import PythonAstREPLTool
from langchain.tools import tool
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from pydantic import BaseModel, Field
from typing import Literal, Optional, List, Dict, Any


/opt/conda/envs/sam/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1) 데이터 생성

- 기간: 2025-01 ~ 2025-03 (일 단위)
- 지역: Seoul, Busan, Daejeon
- 기기 타입: robot_vacuum / washing_machine / air_conditioner
- 펌웨어 버전: 기기 타입별 2~3개
- 지표: power_w, bump_events(로봇), error_code(간헐 발생)

> 이 데이터는 Sub-agent(Data Analyst)의 Python 분석 대상입니다.


In [3]:
rng = np.random.default_rng(7)

# 더미 텔레메트리 데이터 생성 (실데이터 아님)
days = pd.date_range("2025-01-01", "2025-03-31", freq="D")
regions = ["Seoul", "Busan", "Daejeon"]
device_types = ["robot_vacuum", "washing_machine", "air_conditioner"]
fw_versions = {
    "robot_vacuum": ["1.1.0", "1.2.0", "1.2.1"],
    "washing_machine": ["2.1.5", "2.1.7"],
    "air_conditioner": ["3.5.0", "3.5.1", "3.5.2"],
}

rows = []
for d in days:
    for r in regions:
        for dt in device_types:
            # 하루/지역/기기타입별 80~160대 샘플(임의)
            n = int(rng.integers(80, 161))
            for _ in range(n):
                fw = rng.choice(fw_versions[dt])
                # 에러율(임의): 특정 버전에서 washer 5C 증가하도록 의도적으로 bias
                base_err = 0.01
                if dt == "washing_machine" and fw == "2.1.7" and d >= pd.Timestamp("2025-02-10"):
                    base_err = 0.05
                # robot vacuum은 bump 이벤트가 많은 날(주말) 증가시키는 편향
                bump = int(rng.poisson(6 if d.dayofweek >= 5 else 3))
                err_code = None
                if rng.random() < base_err:
                    err_code = "5C" if dt == "washing_machine" else ("E2" if dt == "air_conditioner" else "BUMP_STUCK")

                power_w = float(max(50, rng.normal(200 if dt=="robot_vacuum" else 600 if dt=="air_conditioner" else 400, 80)))
                rows.append({
                    "date": d.strftime("%Y-%m-%d"),
                    "region": r,
                    "device_type": dt,
                    "fw": fw,
                    "power_w": round(power_w, 1),
                    "bump_events": bump if dt=="robot_vacuum" else 0,
                    "error_code": err_code,
                })

df = pd.DataFrame(rows)
df.head()


,date,region,device_type,fw,power_w,bump_events,error_code
0,2025-01-01,Seoul,robot_vacuum,1.2.0,204.8,3,NaN
1,2025-01-01,Seoul,robot_vacuum,1.2.0,125.6,3,NaN
2,2025-01-01,Seoul,robot_vacuum,1.2.1,98.6,6,NaN
3,2025-01-01,Seoul,robot_vacuum,1.1.0,50.0,0,NaN
4,2025-01-01,Seoul,robot_vacuum,1.1.0,121.7,4,NaN


**실행 순서**: 반드시 **위쪽 "1) 데이터 생성" 셀을 먼저 실행**한 뒤 이 셀을 실행하세요.  
`df`가 없으면 `PythonAstREPLTool(locals={"df": df, ...})`에서 `NameError`가 납니다.

## 2) Sub-agents 정의

### (A) Data Analyst
- df를 사용해 수치 계산, 필터링, 이상치 탐지

### (B) Report Writer
- Data Analyst 결과(숫자/표)를 받아 자연어 보고서 작성

### (C) QA Reviewer
- 숫자/논리/누락을 체크하고 개선 지시를 작성


In [4]:
# LLM (역할별로 온도 다르게)


PROXY_URL = "http://10.0.4.135:8000/v1" 
PROXY_TOKEN = "1b2d4df88f5d71a9c55617796faf99f1" 

planner_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, base_url=PROXY_URL, api_key=PROXY_TOKEN)
writer_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3, base_url=PROXY_URL, api_key=PROXY_TOKEN)
qa_llm = ChatOpenAI(model= "gpt-4o-mini", temperature=0, base_url=PROXY_URL, api_key=PROXY_TOKEN)

# Data Analyst는 Python tool 포함
python_tool = PythonAstREPLTool(locals={"df": df, "pd": pd, "np": np})

# create_agent 루프 대신: LLM이 코드 생성 → Python 직접 실행 (재귀 없음)
def data_analyst_agent_fn(query: str) -> str:
    """LLM으로 pandas 코드 생성 후 python_tool로 실행. 결과를 JSON 문자열로 반환."""
    code_prompt = (
        "df, pd, np 가 이미 import/정의되어 있습니다.\n"
        "아래 분석 요청에 맞는 pandas/numpy 코드를 작성해, 마지막 줄에 반드시 print(json.dumps(result, ensure_ascii=False, default=str))로 결과를 출력하세요.\n"
        "result는 dict이고 metrics, notes, evidence 키를 포함해야 합니다.\n"
        "코드만 출력하세요. 설명·마크다운 금지.\n\n"
        f"[분석 요청]\n{query}"
    )
    resp = planner_llm.invoke(code_prompt)
    code = resp.content.strip()
    if code.startswith("```"):
        lines = code.split("\n")
        code = "\n".join(lines[1:-1] if lines[-1].strip() == "```" else lines[1:])
    code = "import json\n" + code
    try:
        result_txt = python_tool.run(code)
    except Exception as e:
        result_txt = json.dumps({"metrics": {}, "notes": f"코드 실행 오류: {e}", "evidence": code[:300]})
    return result_txt or json.dumps({"metrics": {}, "notes": "결과 없음", "evidence": code[:300]})

report_writer_agent = create_agent(
    model=writer_llm,
    tools=[],
    system_prompt=(
        "당신은 제품 텔레메트리 리포트 라이터입니다.\n"
        "- 분석가의 JSON 결과를 읽고, 한국어 보고서로 재구성하세요.\n"
        "- 숫자 근거를 반드시 포함하고, 과장된 단정은 피하세요.\n"
        "- 마지막에 '추가로 확인하면 좋은 데이터'를 2개 제안하세요."
    )
)

qa_reviewer_agent = create_agent(
    model=qa_llm,
    tools=[],
    system_prompt=(
        "당신은 QA 리뷰어입니다.\n"
        "- 보고서에서 숫자/논리/누락/모호한 표현을 찾으세요.\n"
        "- 문제가 있으면 '수정 지시'를 구체적으로 작성하세요.\n"
        "- 문제가 없으면 OK만 반환하세요."
    )
)


## 3) Sub-agent들을 Tool로 래핑

Main Agent는 아래 도구들만 호출할 수 있고, **직접 계산하지 않습니다.**


In [5]:
def _extract_json_from_text(s: str) -> str:
    if not s or not s.strip(): return s
    s = s.strip()
    start = s.find("{")
    if start == -1: return s
    return s[start: s.rfind("}") + 1]

@tool
def ask_data_analyst(query: str) -> str:
    """데이터 분석이 필요할 때 호출합니다. df 기반 분석을 수행하고 JSON 문자열을 반환합니다."""
    print("[1/3] Data Analyst 실행 중...")
    txt = data_analyst_agent_fn(query)
    print("[1/3] Data Analyst 완료.")
    return _extract_json_from_text(txt or "")

@tool
def write_report(analysis_json: str) -> str:
    """분석가 결과(JSON 문자열)를 받아 보고서를 작성합니다."""
    print("[2/3] Report Writer 실행 중...")
    res = report_writer_agent.invoke({"messages": [{"role": "user", "content": analysis_json}]})
    print("[2/3] Report Writer 완료.")
    return res["messages"][-1].content

@tool
def qa_review(report_text: str) -> str:
    """보고서를 QA 검수하고 수정 지시 또는 OK를 반환합니다."""
    print("[3/3] QA Reviewer 실행 중...")
    res = qa_reviewer_agent.invoke({"messages": [{"role": "user", "content": report_text}]})
    print("[3/3] QA Reviewer 완료.")
    return res["messages"][-1].content


## 4) 최종 보고서 스키마 (구조화 출력)

Main Agent의 최종 출력은 아래 스키마를 따라야 합니다.


In [6]:
class KeyMetricEntry(BaseModel):
    """지표 이름과 값 (API strict schema용)"""
    key: str
    value: str

class FinalReport(BaseModel):
    short_answer: str
    key_metrics: List[KeyMetricEntry]
    narrative: str
    qa_result: str
    next_questions: List[str]



## 5) Main Agent(감독관) 구성

규칙:
- 질문을 읽고 필요한 분석을 `ask_data_analyst`에 위임
- 결과를 `write_report`로 보고서화
- `qa_review`로 검수
- 최종 결과를 `FinalReport`로 구조화하여 반환

> 업그레이드 포인트:  
> - 감독관은 직접 계산하지 않고, **분석/작성/검수**를 모두 위임해야 합니다.


In [7]:
main_agent = create_agent(
    model=planner_llm,
    tools=[ask_data_analyst, write_report, qa_review],
    system_prompt=(
        "당신은 감독관(Main Agent)입니다.\n"
        "반드시 다음 순서로 각 도구를 **정확히 한 번씩만** 호출하세요.\n"
        "1) ask_data_analyst(질문) 호출 → 2) write_report(분석 JSON) 호출 → 3) qa_review(보고서 원문) 호출\n"
        "세 번 호출이 끝나면 더 이상 도구를 호출하지 말고, 그 결과를 바탕으로 최종 출력만 FinalReport 스키마로 제출하세요."
    ),
    response_format=ToolStrategy(FinalReport),
)

def run_pipeline(question: str) -> FinalReport:
    res = main_agent.invoke({"messages": [{"role": "user", "content": question}]})
    return res["structured_response"]

# 도구를 정확히 한 번씩만 호출하는 고정 파이프라인 (반복 호출 방지)
def run_pipeline_single_pass(question: str) -> FinalReport:
    analysis = ask_data_analyst.invoke({"query": question})
    report_text = write_report.invoke({"analysis_json": analysis})
    qa_result = qa_review.invoke({"report_text": report_text})
    out = planner_llm.with_structured_output(FinalReport).invoke(
        f"""다음 분석/보고서/QA 결과를 FinalReport 스키마로 정리하세요. key_metrics는 key, value 문자열 쌍 리스트로 넣으세요.\n\n분석(JSON): {analysis}\n\n보고서: {report_text}\n\nQA 결과: {qa_result}"""
    )
    return out


In [8]:
# 예시 (분석 → 보고서 → QA → 구조화)
q = "2025-02-10 이후 세탁기(washing_machine)에서 5C 에러가 증가하는지 확인하고, 특정 펌웨어 버전과의 상관을 근거(지표)와 함께 설명해줘."
print("파이프라인 시작 (진행 단계가 출력됩니다)...")
# Main Agent 사용 (가끔 write_report/qa_review를 여러 번 호출할 수 있음)
# report = run_pipeline(q)
# 각 도구를 정확히 한 번씩만 쓰는 고정 파이프라인
report = run_pipeline_single_pass(q)
report


파이프라인 시작 (진행 단계가 출력됩니다)...
[1/3] Data Analyst 실행 중...
[1/3] Data Analyst 완료.
[2/3] Report Writer 실행 중...
[2/3] Report Writer 완료.
[3/3] QA Reviewer 실행 중...
[3/3] QA Reviewer 완료.


FinalReport(short_answer='분석을 진행할 수 없습니다.', key_metrics=[KeyMetricEntry(key='Error Type', value='KeyError'), KeyMetricEntry(key='Missing Key', value='product')], narrative="제공된 정보에 'product'라는 키가 포함되어 있지 않아 분석을 진행할 수 없습니다. JSON 데이터의 구조나 내용을 확인해 주시면, 그에 맞춰 한국어 보고서를 작성할 수 있도록 도와드리겠습니다. 추가적인 정보나 데이터를 제공해 주시면 감사하겠습니다.", qa_result='OK', next_questions=['JSON 데이터의 구조를 확인할 수 있을까요?', '어떤 추가 정보를 제공할 수 있나요?', '다른 키를 사용하여 분석을 진행할 수 있나요?'])

## 📝 실습 과제

### 문제 1) 에러 복구 루프 추가 (상)
Data Analyst가 가끔 JSON이 아닌 텍스트를 반환하거나, 코드 실행이 실패할 수 있습니다.

- `ask_data_analyst`에서 반환값을 `json.loads()`로 파싱해보고,
  실패하면 **재시도 프롬프트**를 1회 더 보내서 JSON 형태로 고치게 하세요.

### 문제 2) 이상 패턴 탐지 기준 강화 (상)
- 단순히 "가장 큰 값"을 찾는 대신,
  - 특정 날짜 이후 **에러율이 3배 이상 증가**
  - 또는 최근 7일 평균 대비 **+2σ 이상 스파이크**
  같은 기준으로 이상 패턴을 탐지하도록 Data Analyst 시스템 프롬프트를 개선하세요.

### 문제 3) 보고서 품질 QA를 구조화 (중~상)
- QA 결과를 단순 OK 문자열이 아니라,
  - issues(list)
  - severity(enum)
  - fix(list)
  로 구조화해서 FinalReport에 넣도록 확장해보세요.

> 아래 셀은 학생용 TODO입니다.  
> (강사용 답안은 바로 아래에 예시로 제공합니다.)


In [9]:
# ✅ TODO 셀 (학생용)
# - ask_data_analyst를 JSON 파싱 + 1회 재시도로 업그레이드
# - Data Analyst 시스템 프롬프트 개선

# raise NotImplementedError("학생용 TODO 셀입니다. 아래 '참고 답안'을 보기 전까지 구현해보세요.")


In [10]:
# ✅ 참고 답안

def _extract_json_from_text(s: str) -> str:
    """응답에서 JSON 블록만 추출 (마크다운/설명 제거)"""
    if not s or not s.strip():
        return s
    s = s.strip()
    start = s.find("{")
    if start == -1:
        return s
    end = s.rfind("}") + 1
    return s[start:end] if end > start else s

@tool
def ask_data_analyst_v2(query: str) -> str:
    """df 기반 분석을 수행하고 JSON 문자열을 반환합니다. JSON 파싱 실패 시 1회 재시도합니다."""
    txt = _extract_json_from_text(data_analyst_agent_fn(query) or "")
    try:
        json.loads(txt)
        return txt
    except Exception:
        # 재시도: LLM에게 직접 JSON 수정 요청
        retry_prompt = (
            f"아래 텍스트에서 JSON만 추출해 수정하세요. metrics, notes, evidence 키를 포함한 순수 JSON만 출력하세요.\n\n{txt[:800]}"
        )
        resp = planner_llm.invoke(retry_prompt)
        txt2 = _extract_json_from_text(resp.content or "")
        try:
            json.loads(txt2)
            return txt2
        except Exception:
            return json.dumps({"metrics": {}, "notes": "JSON 변환 실패. 원문 요약", "evidence": (txt or txt2)[:500]})

print("✅ 답안 함수 ask_data_analyst_v2 준비 완료") 


✅ 답안 함수 ask_data_analyst_v2 준비 완료


In [11]:
main_agent_v2 = create_agent(
    model=planner_llm,
    tools=[ask_data_analyst_v2, write_report, qa_review],
    system_prompt=(
        "당신은 감독관(Main Agent)입니다. 각 도구를 **정확히 한 번씩만** 호출하세요. "
        "1) ask_data_analyst_v2 → 2) write_report → 3) qa_review. 끝나면 FinalReport로만 제출하세요."
    ),
    response_format=ToolStrategy(FinalReport),
)

def run_pipeline_v2(question: str) -> FinalReport:
    res = main_agent_v2.invoke({"messages": [{"role": "user", "content": question}]})
    return res["structured_response"]

# v2 단일 패스: JSON 재시도 있는 분석가 → 보고서 → QA → 구조화 (각 1회만)
def run_pipeline_v2_single_pass(question: str) -> FinalReport:
    analysis = ask_data_analyst_v2.invoke({"query": question})
    report_text = write_report.invoke({"analysis_json": analysis})
    qa_result = qa_review.invoke({"report_text": report_text})
    out = planner_llm.with_structured_output(FinalReport).invoke(
        f"""다음 분석/보고서/QA 결과를 FinalReport 스키마로 정리하세요. key_metrics는 key, value 문자열 쌍 리스트로 넣으세요.\n\n분석: {analysis}\n\n보고서: {report_text}\n\nQA 결과: {qa_result}"""
    )
    return out


In [12]:
# Main Agent 사용 시 write_report/qa_review가 여러 번 호출될 수 있음 → 단일 패스 권장
report = run_pipeline_v2_single_pass("최근 7일간 세탁기 에러율 추세와 이상 패턴을 분석해서 보고서로 작성해줘.")
print(report)


[2/3] Report Writer 실행 중...
[2/3] Report Writer 완료.
[3/3] QA Reviewer 실행 중...
[3/3] QA Reviewer 완료.
short_answer='분석을 진행할 수 없습니다.' key_metrics=[KeyMetricEntry(key='notes', value='JSON 변환 실패. 원문 유출'), KeyMetricEntry(key='evidence', value='TypeError: Invalid comparison between dtype=str and Timestamp')] narrative='보고서 작성에 필요한 데이터가 부족하여 분석을 진행할 수 없습니다. 제공된 JSON에는 메트릭스(metrics)와 관련된 정보가 포함되어 있지 않으며, 오류 메시지인 "TypeError: Invalid comparison between dtype=str and Timestamp"는 데이터 처리 과정에서 발생한 문제를 나타냅니다. 이와 관련하여 추가로 확인하면 좋은 데이터는 다음과 같습니다: 1. 메트릭스(metrics) 섹션에 포함된 구체적인 수치 데이터. 2. 오류 발생 시점의 데이터 형식 및 구조에 대한 정보.' qa_result='수정 지시: 1. "메트릭스(metrics) 섹션에 포함된 구체적인 수치 데이터"라는 표현이 모호합니다. 어떤 종류의 수치 데이터가 필요한지 구체적으로 명시해 주세요. 예를 들어, 평균, 최대값, 최소값 등 어떤 메트릭스가 필요한지 명확히 하십시오. 2. "오류 발생 시점의 데이터 형식 및 구조에 대한 정보"라는 표현도 모호합니다. 어떤 데이터 형식(예: JSON, CSV 등)과 구조(예: 필드명, 데이터 타입 등)를 확인해야 하는지 구체적으로 설명해 주세요.' next_questions=['메트릭스(metrics) 섹션에 포함된 구체적인 수치 데이터는 무엇인가요?', '오류 발생 시점의 데이터 형식과 구조는 어떻게 되나요?']


In [13]:
print(report.model_dump_json(indent=2, ensure_ascii=False))


{
  "short_answer": "분석을 진행할 수 없습니다.",
  "key_metrics": [
    {
      "key": "notes",
      "value": "JSON 변환 실패. 원문 유출"
    },
    {
      "key": "evidence",
      "value": "TypeError: Invalid comparison between dtype=str and Timestamp"
    }
  ],
  "narrative": "보고서 작성에 필요한 데이터가 부족하여 분석을 진행할 수 없습니다. 제공된 JSON에는 메트릭스(metrics)와 관련된 정보가 포함되어 있지 않으며, 오류 메시지인 \"TypeError: Invalid comparison between dtype=str and Timestamp\"는 데이터 처리 과정에서 발생한 문제를 나타냅니다. 이와 관련하여 추가로 확인하면 좋은 데이터는 다음과 같습니다: 1. 메트릭스(metrics) 섹션에 포함된 구체적인 수치 데이터. 2. 오류 발생 시점의 데이터 형식 및 구조에 대한 정보.",
  "qa_result": "수정 지시: 1. \"메트릭스(metrics) 섹션에 포함된 구체적인 수치 데이터\"라는 표현이 모호합니다. 어떤 종류의 수치 데이터가 필요한지 구체적으로 명시해 주세요. 예를 들어, 평균, 최대값, 최소값 등 어떤 메트릭스가 필요한지 명확히 하십시오. 2. \"오류 발생 시점의 데이터 형식 및 구조에 대한 정보\"라는 표현도 모호합니다. 어떤 데이터 형식(예: JSON, CSV 등)과 구조(예: 필드명, 데이터 타입 등)를 확인해야 하는지 구체적으로 설명해 주세요.",
  "next_questions": [
    "메트릭스(metrics) 섹션에 포함된 구체적인 수치 데이터는 무엇인가요?",
    "오류 발생 시점의 데이터 형식과 구조는 어떻게 되나요?"
  ]
}
